# 🌾 Kisan Shakti — Crop Disease Model Training (Google Colab)
## Transfer Learning using MobileNetV3 for PlantVillage (Tomato, Potato, Corn)

This notebook provides a complete, end-to-end training pipeline for the **Kisan Shakti** agricultural assistance system.

### Complete Workflow:
1. **GPU Check & Environment Setup** (Uses free T4 GPU in Colab)
2. **Dataset Download** (High-speed download of PlantVillage dataset)
3. **Filtering 17 Target Classes** (Tomato, Potato, Corn)
4. **Data Augmentation & Normalization**
5. **Dataset Splits** (70% Train, 15% Validation, 15% Test)
6. **MobileNetV3 Transfer Learning**
7. **Training & Validation Loop** (5 Epochs with AdamW)
8. **Evaluation & Metrics** (Test Accuracy, Precision, Recall, F1)
9. **Export & Automatic Download** of `plant_disease_model.pth` for your local Kisan Shakti backend

### Step 1: GPU Check & Library Installation

In [ ]:
!pip install -q torch torchvision torchaudio scikit-learn pandas matplotlib seaborn pillow tqdm

import os
import sys
import json
import time
import shutil
import zipfile
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# Check CUDA GPU availability
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print("Accelerated GPU Training is ACTIVE!")
else:
    print("WARNING: Running on CPU. Please go to Runtime -> Change runtime type -> T4 GPU for 10x faster training.")

### Step 2: High-Speed Dataset Download
Downloads the official PlantVillage archive directly into the Colab environment.

In [ ]:
# Fast direct download from Mendeley Data CDN (~828MB takes ~15 seconds in Colab)
archive_url = "https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/d5652a28-c1d8-4b76-97f3-72fb80f94efc/file_downloaded"
print("Downloading PlantVillage archive...")
!wget -q --show-progress -O plantvillage.zip "$archive_url"
print("Download complete! Extracting archive...")
!unzip -q plantvillage.zip -d pv_raw/
print("Extraction complete!")

### Step 3: Organize the 17 Target Classes for Kisan Shakti
Filters the raw dataset down to the 17 classes for **Tomato**, **Potato**, and **Corn (Maize)**.

In [ ]:
TARGET_CLASSES = [
    # Tomato (10 classes)
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
    # Potato (3 classes)
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    # Corn / Maize (4 classes)
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy"
]

# Locate extracted folder
raw_root = Path("pv_raw")
# Find directory containing the class subfolders
candidates = list(raw_root.glob("**/Tomato___Early_blight"))
if candidates:
    dataset_source_dir = candidates[0].parent
else:
    dataset_source_dir = raw_root

print(f"Source folder identified: {dataset_source_dir}")

# Copy target classes into dedicated dataset folder
clean_dir = Path("dataset_17_classes")
clean_dir.mkdir(exist_ok=True)

print("\n--- Class Distribution ---")
for target in TARGET_CLASSES:
    src_folder = dataset_source_dir / target
    dst_folder = clean_dir / target
    if src_folder.exists():
        if not dst_folder.exists():
            shutil.copytree(src_folder, dst_folder)
        count = len(list(dst_folder.glob("*")))
        print(f"{target:<45} : {count} images")
    else:
        # Check case-insensitive
        matched = [f for f in dataset_source_dir.iterdir() if f.is_dir() and f.name.lower() == target.lower()]
        if matched:
            if not dst_folder.exists():
                shutil.copytree(matched[0], dst_folder)
            count = len(list(dst_folder.glob("*")))
            print(f"{target:<45} : {count} images")
        else:
            print(f"WARNING: {target} not found!")

print("Dataset filtered and organized successfully!")

### Step 4: Data Augmentation & Train/Val/Test Splits

In [ ]:
# Transformations: Random flip, slight rotation, color jitter for training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load full dataset
full_dataset = datasets.ImageFolder(clean_dir, transform=train_transform)
total_images = len(full_dataset)
print(f"Total images across 17 classes: {total_images}")

# 70% Train, 15% Validation, 15% Test
train_len = int(0.70 * total_images)
val_len = int(0.15 * total_images)
test_len = total_images - train_len - val_len

torch.manual_seed(42)
train_data, val_data, test_data = random_split(full_dataset, [train_len, val_len, test_len])

# Apply deterministic eval transforms to val and test
val_data.dataset.transform = eval_transform
test_data.dataset.transform = eval_transform

BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train set: {len(train_data)} | Val set: {len(val_data)} | Test set: {len(test_data)}")

### Step 5: Initialize MobileNetV3 Transfer Learning Model

In [ ]:
# Load MobileNetV3 Small with pretrained ImageNet weights
weights = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=weights)

# Replace classification head for 17 classes
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, len(TARGET_CLASSES))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

print("MobileNetV3 model configured and ready on GPU!")

### Step 6: Model Training Loop (5 Epochs)
Takes ~2 to 3 minutes on a Colab T4 GPU.

In [ ]:
EPOCHS = 5
best_acc = 0.0
best_model_weights = copy.deepcopy(model.state_dict())

print("Starting Model Training...")
for epoch in range(EPOCHS):
    start_t = time.time()
    
    # --- Training Phase ---
    model.train()
    running_loss, running_correct, total_train = 0.0, 0, 0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_correct += torch.sum(preds == labels.data).item()
        total_train += inputs.size(0)
        
    train_loss = running_loss / total_train
    train_acc = running_correct / total_train
    
    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_correct += torch.sum(preds == labels.data).item()
            total_val += inputs.size(0)
            
    val_loss = val_loss / total_val
    val_acc = val_correct / total_val
    elapsed = time.time() - start_t
    
    print(f"Epoch {epoch+1}/{EPOCHS} [{elapsed:.0f}s] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2%} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2%}")
    
    scheduler.step(val_acc)
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())

print(f"\nTraining Complete! Best Validation Accuracy: {best_acc:.2%}")
model.load_state_dict(best_model_weights)

### Step 7: Final Evaluation on Held-Out Test Set

In [ ]:
model.eval()
all_preds = []
all_targets = []

print("Evaluating on held-out test set...")
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

class_names = [full_dataset.classes[i] for i in range(len(full_dataset.classes))]
print("\n" + "="*60)
print("FINAL HELD-OUT TEST EVALUATION REPORT")
print("="*60)
print(classification_report(all_targets, all_preds, target_names=class_names, digits=4))

test_accuracy = np.mean(np.array(all_preds) == np.array(all_targets))
print(f"Test Set Accuracy: {test_accuracy:.2%}")

### Step 8: Export Model & Download to Your Computer
This cell packages the model and automatically initiates a browser download of `plant_disease_model.pth`.

In [ ]:
from google.colab import files

export_dir = Path("exported_artifacts")
export_dir.mkdir(exist_ok=True)

model_export_path = export_dir / "plant_disease_model.pth"
torch.save(model.state_dict(), model_export_path)
size_mb = model_export_path.stat().st_size / (1024*1024)
print(f"Model saved locally in Colab: {model_export_path} ({size_mb:.2f} MB)")

print("\nTriggering browser download of 'plant_disease_model.pth'...")
try:
    files.download(str(model_export_path))
    print("SUCCESS: Check your browser downloads folder for 'plant_disease_model.pth'!")
except Exception as e:
    print(f"Manual download: Click the folder icon on the left sidebar, open 'exported_artifacts', right-click 'plant_disease_model.pth', and click Download.")